# 10 — Plan AMYs and future climate weather

**Learning objectives:** plan locations × completed source years; inspect the weather jobs; select SSP or RCP settings while preserving the source year.

**Prerequisites:** Python 3.11+ and the smhi2epw development feature branch installed with the tutorials extra. See the repository README for installation and kernel selection.

**Expected runtime:** seconds for this offline planning notebook. No network, JAR, Java, EPW or TMY is needed. No weather generation or climate computation runs here.

**Input provenance and units:** Gothenburg and Stockholm coordinates are illustrative city-centre points in decimal degrees (WGS84). The source years 2022 and 2023 are completed calendar years. Local standard time defaults to UTC+1; daylight saving is ignored. No random sampling is used.

**Outputs:** in-memory compiler plans only. The output directory is a future destination and is not created by planning.


In [ ]:
from smhi2epw import WeatherLocation, plan_amy_batch

locations = [
    WeatherLocation("gothenburg", latitude=57.7089, longitude=11.9746),
    WeatherLocation("stockholm", latitude=59.3293, longitude=18.0686),
]
years = [2022, 2023]
jobs = plan_amy_batch(locations, years, "examples/output/future-weather")
assert len(jobs) == 4
assert {job.year for job in jobs} == {2022, 2023}
[(job.city, job.year) for job in jobs]

## Inspect a planned baseline

Every job uses the existing compiler's weather recovery and source provenance rules. Inspect those settings before requesting downloads. Different years and locations have separate destinations.


In [ ]:
from pathlib import Path

assert len({job.output_path for job in jobs}) == 4
assert all(job.provenance_path == job.output_path + ".json" for job in jobs)
assert Path(jobs[0].output_path).name == "gothenburg_2022.epw"
assert jobs[0].weather_policy == "automatic"
locations[0].summary()

## Add climate scenario selections

Constructing a MorphConfig performs no file access or climate computation. SSP needs the global FWG tool; RCP needs the Europe tool. Version 0.3 of pyfwg uses future period labels 2050 and 2080. Multiple model names form one ensemble. Empty selections use all supported values. Urban heat island adjustment is disabled by this adapter.


In [ ]:
from smhi2epw import MorphConfig

ssp = MorphConfig(
    "tools/FutureWeatherGenerator_v4.0.2.jar",
    scenarios=("ssp245", "ssp585"),
    periods=(2050, 2080),
    models=("CanESM5", "MIROC6"),
)
rcp = MorphConfig(
    "tools/FutureWeatherGenerator_Europe_v2.0.2.jar",
    family="rcp",
    scenarios=("rcp45",),
    periods=(2050,),
    models=("ICHEC_EC_EARTH_SMHI_RCA4",),
)
expected_future_files = len(jobs) * (
    len(ssp.scenarios) * len(ssp.periods) + len(rcp.scenarios) * len(rcp.periods)
)
assert expected_future_files == 20
print(ssp)
print(rcp)
print(f"Planned: {len(jobs)} baselines and {expected_future_files} future EPWs")

## Execute after supplying the prerequisites

The following call is shown for a separate, explicitly requested live run. It is not executed by this offline notebook:

```python
from smhi2epw import compile_amy_batch
result = compile_amy_batch(
    locations, years, "examples/output/future-weather",
    morphing=[ssp, rcp],
)
print(result.summary())
```

Install the morphing extra, supply both JARs and Java, and allow provider access before executing it. It creates EPWs, JSON receipts and a progress manifest. Existing outputs are refused; choose a fresh directory for a new run. The manifest records a later failure while retaining earlier completed files.

The baseline name `gothenburg_2022.epw` becomes, for example, `gothenburg_2022_ssp245_2050.epw`. Its source calendar remains distinct from the future climate period. Morphing a hot historical year does not make it a typical future year, and the wrapper does not adjust FWG climate deltas to each AMY's own baseline period. Review that reference period before using absolute future conditions.

**Verification boundary:** this notebook checks planning, not real SMHI acquisition or FWG climate generation. The separately marked FWG tests require real JARs and Java.

## Try it

Add a third location with supplied coordinates and replan. Check that the job count equals the number of locations multiplied by the number of years. Change the selected future scenarios and calculate the expected file count before running a live campaign.

## Key takeaways

- Plan the location/year product offline before downloading weather.
- Keep source years and climate periods in separate fields and filenames.
- Review baseline recovery receipts and climate reference assumptions.
- A valid output file establishes structural validity; scientific climate accuracy needs separate evidence.
